# Step 5. Measure vertebral slip and grade spondylolisthesis

Orders the detected vertebrae from caudal to cranial and, for each adjacent pair, extends the posterior border of the upper vertebra until it meets the endplate of the vertebra below. The slip is the distance from that point to the posterior corner of the lower vertebra, expressed as a percentage of the lower endplate width, and is graded with the Meyerding scale (up to 10% normal; grade I up to 25%; II up to 50%; III up to 75%; IV up to 100%; V above 100%).

**Input:** image folder and the JSON annotations from step 3. **Output:** slip percentage and grade for each vertebral level, drawn on the image.

Code comments are in Spanish.

In [ ]:
# Montar el Drive de Google
from google.colab import drive
drive.mount('/content/drive')

**1. Importar Librerias**

In [ ]:
#Importar las librerias necesariass
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import os
from google.colab import drive
import ipywidgets as widgets
from IPython.display import display

# Configuración de matplotlib para Google Colab
%matplotlib inline

**2. Carga de Archivos**

In [ ]:
def cargar_keypoints(json_folder, image_filename):
    base_name = os.path.basename(image_filename)
    name_without_extension = os.path.splitext(base_name)[0]
    json_path = os.path.join(json_folder, f"{name_without_extension}.json")

    if not os.path.exists(json_path):
        raise FileNotFoundError(f"No se encontró el archivo JSON para {json_path}")
    with open(json_path, 'r') as f:
        keypoints = json.load(f)

    # Ordenar las vértebras en orden descendente
    keypoints['keypoints'] = sorted(keypoints['keypoints'], key=lambda v: np.mean([p[1] for p in v]), reverse=True)
    # Reordenar los puntos clave según el nuevo formato
    for vertebra in keypoints['keypoints']:
        vertebra[0], vertebra[1], vertebra[2], vertebra[3] = vertebra[2], vertebra[0], vertebra[1], vertebra[3]
    return keypoints

**3. Visualización de la Imagen con anotaciones + Calculo de porcentaje de traslación**

In [ ]:
# Función para mostrar la imagen con keypoints
def imagen_conkp(image_path, keypoints, results, lines, divisions):
    plt.figure(figsize=(10, 10))
    img = mpimg.imread(image_path)
    plt.imshow(img)

    keypoints_list = keypoints.get('keypoints', []) #Obtiene los keypoints como lista
    colors = ['cyan', 'magenta', 'yellow', 'green']

    for idx, keypoint_set in enumerate(keypoints_list):
        # Extracción de coordenadas x e y de los keypoints
        x_coords, y_coords = zip(*[(p[0], p[1]) for p in keypoint_set])
        plt.scatter(x_coords, y_coords, color=colors[:len(x_coords)], s=10) #Dibuja los keypoints en la imagen, S para cambiar el tamaño de los puntos
        for i, (x, y) in enumerate(zip(x_coords, y_coords)):
            plt.text(x, y, f'{i+1}', fontsize=8, color=colors[i]) #Agrega el número de keypoint a la imagen
        plt.plot(x_coords + (x_coords[0],), y_coords + (y_coords[0],), color='white', linestyle='-', linewidth=1) #Dibuja los contornos de la vertebra
        plt.text(np.mean(x_coords), np.mean(y_coords), f"V{idx + 1}", fontsize=8, color='white', ha='center') #Agrega el número de vertebra a la imagen

     # Dibujo de las líneas de medición y porcentajes en la imagen
    for line in lines:
        plt.plot([line[0][0], line[1][0]], [line[0][1], line[1][1]], color='brown', linestyle='--', linewidth=1)
    for division in divisions:
        x_coords, y_coords = zip(*division)
        plt.plot(x_coords, y_coords, color='brown', linestyle='-', linewidth=1)
        for i, (x, y) in enumerate(zip(x_coords, y_coords)):
            plt.text(x, y, f'{100 - i*25}%', fontsize=8, color='brown', ha='center') #Dibuja los porcentajes desde 100% a 0% en intervalos de 25

    plt.show()

# Función para calcular la intersección de dos líneas
def calcular_interseccion(line1, line2):
    # Extracción de coordenadas de las líneas
    x1, y1 = line1[0]
    x2, y2 = line1[1]
    x3, y3 = line2[0]
    x4, y4 = line2[1]

    denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4) #Calcula el denominador de la fórmula de intersección. Si es 0, las líneas son paralelas y no se intersectan
    if denom == 0:
        return None

    inter_x = ((x1 * y2 - y1 * x2) * (x3 - x4) - (x1 - x2) * (x3 * y4 - y3 * x4)) / denom
    inter_y = ((x1 * y2 - y1 * x2) * (y3 - y4) - (y1 - y2) * (x3 * y4 - y3 * x4)) / denom

#inter_x e inter_y calculan las coordenadas de la intersección

    return round(inter_x, 2), round(inter_y, 2)

# Función para calcular y clasificar el grado de espondilolistesis
def calculo_espondilo(keypoints):
    vertebrae = keypoints.get('keypoints', [])
    if len(vertebrae) < 2:
        print("No hay suficientes vértebras para calcular espondilolistesis") #se requieren almenos 2 vertebras para realizar el calculo
        return [], [], []

    results, lines, divisions = [], [], []
    for i in range(min(4, len(vertebrae))):
        current = vertebrae[i]
        if i < len(vertebrae) - 1:
            above = vertebrae[i + 1]
        #identifica la vertebra actual y la superior a esta
            current_bottom_right = current[3] #esquina inferior derecha de la vertebra inferior

            point_3_above, point_4_above = above[2], above[3] #esquinas superior/inferior derecha de la vertebra superior
            #dibuja y extiende en ambas direcciones (hacia arriba y abajo) las lineas punteadas en el borde derecho de las vertebras
            extended_line_forward = [(point_3_above[0], point_3_above[1]),
                                     (point_4_above[0] + (point_4_above[0] - point_3_above[0]) * 0.5,
                                      point_4_above[1] + (point_4_above[1] - point_3_above[1]) * 0.5)]
            extended_line_backward = [(point_3_above[0], point_3_above[1]),
                                      (point_3_above[0] - (point_4_above[0] - point_3_above[0]) * 0.5,
                                       point_3_above[1] - (point_4_above[1] - point_3_above[1]) * 0.5)]
            lines.append(extended_line_forward)
            lines.append(extended_line_backward)



            reference_line = [(current[0][0], current[0][1]), (current[3][0], current[3][1])] #define la linea de referencia para la medicion
            interseccion = calcular_interseccion(extended_line_forward, reference_line) #encuentra las coordenadas de la interseccion entre la linea punteada y la linea de referencia (donde estan los %)

            if interseccion:
                interseccion_x, interseccion_y = interseccion
            else:
                interseccion_x, interseccion_y = point_3_above[:2]

            print(f"Intersección para V{i+1}: ({interseccion_x}, {interseccion_y})") #imprime las coordenadas de la interseccion para cada vertebra
          #Mide el desplazamiento de la vertebra
            displacement = np.linalg.norm(np.array([interseccion_x, interseccion_y]) - np.array(current_bottom_right[:2])) #calcula el desplazamiento entre la interseccion y la esquina inferior derecha de la vertebra
            width = np.linalg.norm(np.array(current[0][:2]) - np.array(current[3][:2])) #calcula el ancho de la vertebra
            percentage_slip = (displacement / width) * 100 #calcula el porcentaje de deslizamiento

            #condiciones para clasificacion de meyerding
            grade = ("Negativo" if percentage_slip < 0 else
                     "Normal" if percentage_slip <= 10 else
                     "Grado I" if percentage_slip <= 25 else
                     "Grado II" if percentage_slip <= 50 else
                     "Grado III" if percentage_slip <= 75 else
                     "Grado IV" if percentage_slip <= 100 else
                     "Grado V")

            results.append(f"Entre V{i + 1} y V{i + 2}: Desplazamiento = {displacement:.2f} píxeles, Porcentaje de deslizamiento = {percentage_slip:.2f}%, Clasificación = {grade}")

        #la misma linea punteada pero incluyendo V1
        if i == 0:
            point_3 = current[2]
            point_4 = current[3]
            extended_line_forward_v1 = [(point_3[0], point_3[1]),
                                        (point_4[0] + (point_4[0] - point_3[0]) * 0.5,
                                         point_4[1] + (point_4[1] - point_3[1]) * 0.5)]
            extended_line_backward_v1 = [(point_3[0], point_3[1]),
                                         (point_3[0] - (point_4[0] - point_3[0]) * 0.5,
                                          point_3[1] - (point_4[1] - point_3[1]) * 0.5)]
            lines.append(extended_line_forward_v1)
            lines.append(extended_line_backward_v1)

        division = [(current[0][0] + (current[3][0] - current[0][0]) * j / 4,
                     current[0][1] + (current[3][1] - current[0][1]) * j / 4) for j in range(5)]
        divisions.append(division)

        if i > 0:
            division_inferior = [(current[1][0] + (current[2][0] - current[1][0]) * j / 4,
                                  current[1][1] + (current[2][1] - current[1][1]) * j / 4) for j in range(5)]
            divisions.append(division_inferior)

        print(results[-1]) #imprime el resultado del cálculo del grado de espondilolistesis para la vértebra actual

    return results, lines, divisions

**4. Selección del archivo a visualizar**

In [ ]:
# Establece las rutas de las carpetas donde se encuentran los archivos JPG y JSON
image_folder = ('CARPETA_IMAGENES') #Ingresar la ruta de la carpeta correspondiente a las imagenes
json_folder = ('CARPETA_ANOTACIONES_AUTOMATICAS')  #Ingresar la ruta de la carpeta correspondiente a las anotaciones

# Lista de archivos JPG en la carpeta de imágenes
image_files = [os.path.join(image_folder, f) for f in os.listdir(image_folder) if f.endswith('.jpg') or f.endswith('.jpeg') or f.endswith('.png')]

# Widget interactivo para seleccionar una imagen
dropdown = widgets.Dropdown(
    options=image_files,
    description='Imagen:',
    disabled=False,
)

def visualizar_imagen(image_path):
    try:
        keypoints = cargar_keypoints(json_folder, image_path)
        results, lines, divisions = calculo_espondilo(keypoints)
        imagen_conkp(image_path, keypoints, results, lines, divisions)
    except FileNotFoundError as e:
        print(e)
    except json.JSONDecodeError as e:
        print(f"Error al decodificar JSON: {e}")
    except Exception as e:
        print(f"Ocurrió un error: {e}")

# Botón para ejecutar la visualización de la imagen seleccionada
button = widgets.Button(description="Visualizar")

def on_button_clicked(b):
    visualizar_imagen(dropdown.value)

button.on_click(on_button_clicked)

# Mostrar los widgets
display(dropdown, button)

Intersección para V1: (692.48, 1765.46)
Entre V1 y V2: Desplazamiento = 88.72 píxeles, Porcentaje de deslizamiento = 36.03%, Clasificación = Grado II
Intersección para V2: (587.2, 1567.78)
Entre V2 y V3: Desplazamiento = 18.95 píxeles, Porcentaje de deslizamiento = 7.67%, Clasificación = Normal
Intersección para V3: (539.93, 1354.61)
Entre V3 y V4: Desplazamiento = 10.14 píxeles, Porcentaje de deslizamiento = 4.22%, Clasificación = Normal
Intersección para V4: (549.56, 1105.29)
Entre V4 y V5: Desplazamiento = 1.51 píxeles, Porcentaje de deslizamiento = 0.57%, Clasificación = Normal